# Real Estate Data Analysis -- Bangalore
### Internship Project | Python EDA & Visualisation
---
**Dataset:** 500 residential properties across Bangalore  
**Columns:** Price (Lakhs), Area (SqFt), Bedrooms, Bathrooms, Location, Property Type, Furnishing, Parking, Year Built  
**Goal:** Uncover pricing patterns, demand trends and market insights.

---
## Table of Contents
1. Import Libraries  
2. Load & Inspect Data  
3. Data Cleaning  
4. Feature Engineering  
5. Exploratory Data Analysis (EDA)  
6. Visualisations  
7. Key Business Insights  
8. Conclusion

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import datetime

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi']       = 120
plt.rcParams['font.family']      = 'DejaVu Sans'
plt.rcParams['axes.titlesize']   = 14
plt.rcParams['axes.titleweight'] = 'bold'

print('All libraries imported successfully.')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('cleaned_real_estate.csv')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
for col in ['location','property_type','furnishing','city']:
    print(f'{col:15s}: {df[col].nunique()} unique -> {df[col].unique().tolist()}')

## 3. Data Cleaning

In [ ]:
# 3.1 Missing values
print('Missing values per column:')
print(df.isnull().sum())

num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('Missing values handled.')

In [ ]:
# 3.2 Duplicate rows
print(f'Duplicate rows found: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Remaining rows: {len(df)}')

In [ ]:
# 3.3 Outlier detection (IQR method)
def flag_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return series[(series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)]

for col in ['price_lakhs', 'area_sqft']:
    o = flag_outliers(df[col])
    print(f'{col}: {len(o)} outlier(s) | range -> {o.min():.1f} to {o.max():.1f}')

In [ ]:
# 3.4 Remove invalid rows
before = len(df)
df = df[(df['price_lakhs'] > 0) & (df['area_sqft'] > 0)]
df = df[df['bedrooms'] <= 10]
print(f'Removed {before - len(df)} invalid rows. Final dataset: {len(df)} rows.')

## 4. Feature Engineering

In [ ]:
# Price per square foot (in Rs)
df['price_per_sqft'] = (df['price_lakhs'] * 100000 / df['area_sqft']).round(2)

# Property age
df['property_age'] = datetime.date.today().year - df['year_built']

# Price segment
def segment(p):
    if p < 50:    return 'Affordable'
    elif p < 100: return 'Mid-Range'
    elif p < 150: return 'Premium'
    else:         return 'Luxury'
df['price_segment'] = df['price_lakhs'].apply(segment)

# BHK label
df['bhk'] = df['bedrooms'].astype(str) + ' BHK'

print('New features:')
df[['price_lakhs','price_per_sqft','property_age','price_segment','bhk']].head(8)

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Price statistics
stats = df['price_lakhs'].agg(['count','mean','median','std','min','max'])
for k, v in stats.items():
    print(f'  {k:8s}: Rs {v:,.2f} L')

In [ ]:
# Property type breakdown
pt = df.groupby('property_type')['price_lakhs'].agg(['count','mean','min','max']).round(2)
pt.columns = ['Count','Avg Price (L)','Min Price (L)','Max Price (L)']
pt['Market Share %'] = (pt['Count'] / pt['Count'].sum() * 100).round(1)
print(pt.sort_values('Avg Price (L)', ascending=False).to_string())

In [ ]:
# Top 10 locations by average price
loc_price = (df.groupby('location')['price_lakhs']
               .agg(['mean','count','min','max'])
               .round(2)
               .sort_values('mean', ascending=False))
loc_price.columns = ['Avg Price (L)','Listings','Min','Max']
print(loc_price.head(10).to_string())

In [ ]:
# BHK analysis
bhk = df.groupby('bedrooms').agg(
    Count     = ('price_lakhs','count'),
    Avg_Price = ('price_lakhs','mean'),
    Avg_Area  = ('area_sqft','mean')
).round(2)
bhk['Share %'] = (bhk['Count'] / bhk['Count'].sum() * 100).round(1)
print(bhk.to_string())

In [ ]:
# Correlation matrix
corr_cols = ['price_lakhs','area_sqft','bedrooms','bathrooms','parking','property_age']
corr = df[corr_cols].corr().round(3)
print('Correlation with price_lakhs:')
print(corr['price_lakhs'].sort_values(ascending=False).to_string())

In [ ]:
# Furnishing effect
furn = df.groupby('furnishing')['price_lakhs'].agg(['mean','count']).round(2)
furn.columns = ['Avg Price (L)', 'Count']
print(furn.sort_values('Avg Price (L)', ascending=False).to_string())

## 6. Visualisations

### 6.1 Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Property Price Distribution -- Bangalore', fontsize=15, fontweight='bold')

axes[0].hist(df['price_lakhs'], bins=25, color='#1f6fad', edgecolor='white', linewidth=0.6)
axes[0].axvline(df['price_lakhs'].mean(),   color='orange', lw=2, ls='--',
                label=f"Mean: Rs{df['price_lakhs'].mean():.1f}L")
axes[0].axvline(df['price_lakhs'].median(), color='red',    lw=2, ls=':',
                label=f"Median: Rs{df['price_lakhs'].median():.1f}L")
axes[0].set_xlabel('Price (Lakhs)')
axes[0].set_ylabel('Number of Properties')
axes[0].set_title('Histogram')
axes[0].legend()

axes[1].boxplot(df['price_lakhs'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#aed6f1', color='#1a5276'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('Price (Lakhs)')
axes[1].set_title('Box Plot (Outlier View)')
axes[1].set_xticklabels(['Price (L)'])

plt.tight_layout()
plt.savefig('plot_01_price_distribution.png', bbox_inches='tight')
plt.show()
print('Insight: Prices are right-skewed -- most properties are in the 40-120L range.')

### 6.2 Property Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Property Type Analysis', fontsize=15, fontweight='bold')
colors = ['#1f6fad','#2e86c1','#5dade2']

counts = df['property_type'].value_counts()
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 2, str(val), ha='center', fontweight='bold')
axes[0].set_title('Number of Listings')
axes[0].set_xlabel('Property Type')
axes[0].set_ylabel('Count')

avg_price = df.groupby('property_type')['price_lakhs'].mean().sort_values(ascending=False)
bars = axes[1].bar(avg_price.index, avg_price.values, color=colors, edgecolor='white')
for bar, val in zip(bars, avg_price.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'Rs{val:.1f}L', ha='center', fontweight='bold')
axes[1].set_title('Average Price (Lakhs)')
axes[1].set_xlabel('Property Type')
axes[1].set_ylabel('Avg Price (Lakhs)')

plt.tight_layout()
plt.savefig('plot_02_property_type.png', bbox_inches='tight')
plt.show()

### 6.3 Top 10 Locations

In [ ]:
top_loc = df.groupby('location')['price_lakhs'].mean().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_loc.index[::-1], top_loc.values[::-1],
               color=sns.color_palette('Blues_d', len(top_loc)), edgecolor='white')
for bar, val in zip(bars, top_loc.values[::-1]):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'Rs{val:.1f}L', va='center', fontweight='bold')
ax.set_xlabel('Average Price (Lakhs)')
ax.set_title('Top 10 Locations by Average Price -- Bangalore')
ax.set_xlim(0, top_loc.max() * 1.18)
plt.tight_layout()
plt.savefig('plot_03_top_locations.png', bbox_inches='tight')
plt.show()

### 6.4 BHK Configuration

In [ ]:
bhk_count = df['bedrooms'].value_counts().sort_index()
bhk_price = df.groupby('bedrooms')['price_lakhs'].mean().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bedroom (BHK) Analysis', fontsize=15, fontweight='bold')

axes[0].pie(bhk_count.values,
            labels=[f'{b} BHK' for b in bhk_count.index],
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Blues', len(bhk_count)),
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[0].set_title('Market Share by BHK')

axes[1].bar([f'{b} BHK' for b in bhk_price.index], bhk_price.values,
            color=sns.color_palette('Blues_d', len(bhk_price)), edgecolor='white')
for i, val in enumerate(bhk_price.values):
    axes[1].text(i, val+1, f'Rs{val:.1f}L', ha='center', fontweight='bold')
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Average Price (Lakhs)')
axes[1].set_title('Average Price by BHK')

plt.tight_layout()
plt.savefig('plot_04_bhk_analysis.png', bbox_inches='tight')
plt.show()

### 6.5 Furnishing Impact

In [ ]:
furn_price = df.groupby('furnishing')['price_lakhs'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors_f = ['#1f6fad', '#5dade2', '#aed6f1']
bars = ax.bar(furn_price.index, furn_price.values, color=colors_f, edgecolor='white', width=0.5)
for bar, val in zip(bars, furn_price.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'Rs{val:.1f}L', ha='center', fontweight='bold', fontsize=11)
ax.set_xlabel('Furnishing Status')
ax.set_ylabel('Average Price (Lakhs)')
ax.set_title('Effect of Furnishing on Average Price')
plt.tight_layout()
plt.savefig('plot_05_furnishing.png', bbox_inches='tight')
plt.show()

### 6.6 Area vs Price Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'Villa':'#1f6fad','Apartment':'#e74c3c','Independent House':'#27ae60'}
for ptype, grp in df.groupby('property_type'):
    ax.scatter(grp['area_sqft'], grp['price_lakhs'],
               c=colors_map[ptype], label=ptype, alpha=0.6, edgecolors='none', s=40)

m, b = np.polyfit(df['area_sqft'], df['price_lakhs'], 1)
x_line = np.linspace(df['area_sqft'].min(), df['area_sqft'].max(), 100)
ax.plot(x_line, m*x_line+b, color='black', lw=1.5, ls='--', label='Trend Line')

ax.set_xlabel('Area (SqFt)')
ax.set_ylabel('Price (Lakhs)')
ax.set_title('Area vs Price -- Coloured by Property Type')
ax.legend(title='Property Type')
plt.tight_layout()
plt.savefig('plot_06_area_vs_price.png', bbox_inches='tight')
plt.show()
print(f"Correlation (Area vs Price): {df['area_sqft'].corr(df['price_lakhs']):.3f}")

### 6.7 Correlation Heatmap

In [ ]:
corr_cols   = ['price_lakhs','area_sqft','bedrooms','bathrooms','parking','property_age','price_per_sqft']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, linewidths=0.5, annot_kws={'size': 10}, ax=ax)
ax.set_title('Correlation Heatmap -- Numeric Features')
plt.tight_layout()
plt.savefig('plot_07_correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 6.8 Price Segmentation

In [ ]:
seg_order  = ['Affordable', 'Mid-Range', 'Premium', 'Luxury']
seg_counts = df['price_segment'].value_counts().reindex(seg_order)
colors_seg = ['#aed6f1','#5dade2','#2980b9','#1a5276']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Price Segmentation of Bangalore Properties', fontsize=15, fontweight='bold')

axes[0].bar(seg_counts.index, seg_counts.values, color=colors_seg, edgecolor='white')
for i, val in enumerate(seg_counts.values):
    axes[0].text(i, val+2, str(val), ha='center', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_title('Properties per Segment')

axes[1].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
            colors=colors_seg, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Market Share by Segment')

plt.tight_layout()
plt.savefig('plot_08_price_segmentation.png', bbox_inches='tight')
plt.show()

### 6.9 Construction Trend by Year

In [ ]:
year_count = df['year_built'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(year_count.index, year_count.values, alpha=0.3, color='#1f6fad')
ax.plot(year_count.index, year_count.values, color='#1f6fad', lw=2, marker='o', markersize=4)
ax.set_xlabel('Year Built')
ax.set_ylabel('Number of Properties')
ax.set_title('Real Estate Construction Trend (1995 - 2024)')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('plot_09_construction_trend.png', bbox_inches='tight')
plt.show()
print('Insight: Construction surged post-2015, reflecting growing demand.')

## 7. Key Business Insights

In [ ]:
print('=' * 58)
print('  KEY BUSINESS INSIGHTS -- BANGALORE REAL ESTATE')
print('=' * 58)

avg_p   = round(df['price_lakhs'].mean(), 1)
med_p   = round(df['price_lakhs'].median(), 1)
avg_psf = round(df['price_per_sqft'].mean(), 0)
top_loc = df.groupby('location')['price_lakhs'].mean().idxmax()
pop_loc = df['location'].mode()[0]
top_pt  = df.groupby('property_type')['price_lakhs'].mean().idxmax()
top_bhk = df.groupby('bedrooms')['price_lakhs'].mean().idxmax()
pop_bhk = df['bedrooms'].mode()[0]
top_fur = df.groupby('furnishing')['price_lakhs'].mean().idxmax()
avg_age = round(df['property_age'].mean(), 0)

print(f'''
DATASET  : {len(df)} properties | City: Bangalore
TYPES    : Villa, Apartment, Independent House

PRICING
  Average Price       : Rs {avg_p} Lakhs
  Median Price        : Rs {med_p} Lakhs
  Avg Price / SqFt    : Rs {int(avg_psf):,}

PROPERTY TYPE
  Highest Avg Price   : {top_pt}

LOCATION
  Most Listings       : {pop_loc}
  Most Expensive      : {top_loc}

CONFIGURATION
  Most Popular BHK    : {pop_bhk} BHK
  Highest Priced BHK  : {top_bhk} BHK

FURNISHING
  Premium Choice      : {top_fur}

AGE
  Avg Property Age    : {int(avg_age)} years
  Year Range          : {df['year_built'].min()} - {df['year_built'].max()}
''')
print('=' * 58)

## 8. Conclusion

This analysis of **500 Bangalore real estate listings** reveals the following key takeaways:

| Finding | Detail |
|---|---|
| **Price Range** | Rs 25L to Rs 181L; most properties sit in the Rs 50-120L bracket |
| **Best Value** | Apartments in Electronic City and Yelahanka offer the lowest price/sqft |
| **Premium Zones** | Indiranagar and Whitefield command significantly higher prices |
| **Most in Demand** | 2 BHK Apartments are the most popular configuration |
| **Furnishing Premium** | Furnished properties are priced higher on average |
| **Construction Boom** | Post-2015 saw a strong surge in new developments |

> **Next Steps:** Build a machine learning model (Linear Regression / Random Forest)
> to predict property prices based on the features explored above.

---
*Project completed as part of Data Analysis Internship -- Bangalore Real Estate Study*